# K-Nearest Neighbors (KNN) Teaching Module

-- By GPT

**Audience:** Students with basic Python, NumPy, Matplotlib, and introductory machine learning background.

## Learning Goals

By the end of this notebook, students should be able to:

- Explain the basic idea of K-nearest neighbors
- Use KNN for classification
- Understand the role of the parameter `k`
- Understand why feature scaling matters
- Visualize KNN decision boundaries
- Evaluate model accuracy
- Use KNN for regression

---

#### <p style="text-align: right;"> &#9989; **put your name here** </p>

---
## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, mean_squared_error

np.random.seed(0)

---

## 1. What Is KNN?

**K-nearest neighbors (KNN)** is a simple machine learning method.

For a new data point, KNN:

1. Finds the `k` closest training points
2. Looks at their labels
3. Predicts the majority label for classification

KNN is called a **lazy learning method** because it does not build an explicit model during training. It stores the training data and uses it during prediction.


---

## 2. A Simple Classification Dataset

We create a 2D dataset so we can visualize the result. Here, we use `sklearn` library.

- `make_classification`: create a synthetic dataset that contains data points based on out need.
- Each 2D data point contains **two** features: ($x_1, x_2$) and **one** label.

In [ ]:
X, y = make_classification(
    n_samples=300,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    class_sep=1.2,
    random_state=0)

plt.figure(figsize=(6, 5))
scatter = plt.scatter(X[:,0], X[:,1], c=y)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Synthetic Classification Dataset")
plt.axis('equal')
plt.grid(True)
plt.legend(*scatter.legend_elements(), title="Class")
plt.show()


&#9989; Do This -- Do a search and answer these questions.
- What does `n_redundant` do?
- What does `n_n_informative` do?
- What are the data type and dimensions of `X` and `y`?

## 3. Train/Test Split

We split the data into training and testing sets.

The model learns from the training set and is evaluated on the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=0)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

---

### Euclidean distance between 2 points

$$ d_{2-1} = \big|\big|\vec{r_1} - \vec{r_2}\big|\big|_2 = \sqrt{\big(x_1 - x_2\big)^2 + \big(y_1 - y_2\big)^2}$$  

&#9989; Do This --
- Write a code to calculate the distannces of a point $(1,0)$ to all the training data points.
- List the positions and labels of the closest five points.
  

In [ ]:
# =====================================================
# Compute Euclidean distances
# =====================================================
my_point = ??


distances = []
for i in range(len(X_train)):

    x_i = X_train[i]

    # Euclidean distance
    d = ??

    distances.append(d)

distances = np.array(distances)


print("Distances:")
for i in range(len(X_train)):
    print(
        f"Point {X_train[i]} "
        f"Label={y_train[i]} "
        f"Distance={distances[i]:.3f}")

# =====================================================
# Find k nearest neighbors
# =====================================================

k = 5

# sort indices from smallest distance
sorted_indices = ??

print("\nSorted indices:")
print(sorted_indices)

# nearest neighbors
k_indices = sorted_indices[??]

print("\nNearest neighbors:")
print(k_indices)

# corresponding labels
k_labels = y_train[k_indices]

print("\nNeighbor labels:")
print(k_labels)


plt.figure(figsize=(6, 5))
plt.scatter(X_train[:,0], X_train[:,1],c=y_train)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("train data")
plt.axis('equal')
plt.grid(True)

plt.axis('equal')
plt.scatter(my_point[0,0], my_point[0,1], s=80)
plt.scatter(X_train[k_indices,0], X_train[k_indices,1], facecolors='none',edgecolors='black',s=60)

plt.show()



### Classifying by majority vote

Based on the K-nearest neighbors, we classify the new point based on the majority of its neighbors. 

In [ ]:
# =====================================================
# Majority vote
# =====================================================

unique_labels, counts = np.unique(
    k_labels,
    return_counts=True)

print("\nunique labels and counts:")
print(unique_labels, counts)

prediction = unique_labels[np.argmax(counts)]

print("\nPrediction:")
print(prediction)

---

## 4. Feature Scaling

KNN is based on distance. Therefore, feature scaling is very important.

If one feature has much larger numerical values than another, it can dominate the distance calculation.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

---

## 5. Train a KNN Classifier

Here we use `k = 5`, meaning each prediction depends on the 5 nearest neighbors.


In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
print("Test accuracy:", acc)

---

## 6. Confusion Matrix

A confusion matrix shows correct and incorrect predictions by class.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("KNN Confusion Matrix")
plt.show()

---

## 7. Visualizing the Decision Boundary

For a 2D dataset, we can show the regions where the classifier predicts each class.


In [ ]:
def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300))

    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid_points)
    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, Z, alpha=0.3)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.title(title)
    plt.grid(True)
    plt.show()

plot_decision_boundary(knn, X_train_scaled, y_train, "KNN Decision Boundary, k = 5")

---

## 8. Effect of Different k Values

Next, let's examine the effects of choosing different number of nearest neighbors. Run values of k through 1, 3, 15, 30, 50.


<!-- Small `k` values can create very flexible decision boundaries.
Large `k` values create smoother decision boundaries but may underfit. -->


In [ ]:
for k in [??]:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    plot_decision_boundary(model, X_train_scaled, y_train, f"KNN Decision Boundary, k = {k}")

&#9989; Do This -- Answer the these questions.
- What do you observe the change of decision boundaries with increasing `k` value?
- For a overlapped dataset, larger or smaller `k` value will do a better job?

---

## 9. Choosing k Using Test Accuracy

We try several values of `k` and compare test accuracy.


In [ ]:
k_values = range(1, 31)
accuracies = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    y_pred_k = model.predict(X_test_scaled)
    accuracies.append(accuracy_score(y_test, y_pred_k))

plt.figure(figsize=(6, 5))
plt.plot(k_values, accuracies, marker="o")
plt.xlabel("k")
plt.ylabel("Test accuracy")
plt.title("Choosing k for KNN")
plt.grid(True)
plt.show()

best_k = list(k_values)[np.argmax(accuracies)]
print("Best k:", best_k)
print("Best accuracy:", max(accuracies))

---

## 10. KNN on the Iris (鳶尾花) Dataset

Now we apply KNN to a real dataset.

<img src="https://miro.medium.com/v2/resize:fit:720/format:webp/0*SHhnoaaIm36pc1bd" width="640">

The Iris dataset has 4 features:

- sepal length
- sepal width
- petal length
- petal width

There are 3 classes of Iris flowers.


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print("Feature names:")
for name in iris.feature_names:
    print("-", name)

print("Class names:", iris.target_names)

Examine the downloaded Iris data. Plot the data points in 3D.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris,
    test_size=0.25,
    random_state=0,
    stratify=y_iris)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_iris = KNeighborsClassifier(n_neighbors=5)
knn_iris.fit(X_train_scaled, y_train)

y_pred = knn_iris.predict(X_test_scaled)

print("Iris test accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=iris.target_names
)
disp.plot()
plt.title("KNN on Iris Dataset")
plt.show()

---

## 11. KNN Regression

KNN can also be used for regression. which predicts the average value of the nearest neighbors. Instead of voting for a class, it averages the values of nearby points.


In [ ]:
from sklearn.datasets import make_regression

In [ ]:
X_reg, y_reg = make_regression(
    n_samples=200,
    n_features=1,
    noise=20,
    random_state=0)

X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg,
    test_size=0.25,
    random_state=0)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_reg = KNeighborsRegressor(n_neighbors=5)
knn_reg.fit(X_train_scaled, y_train)

y_pred = knn_reg.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
print("Mean squared error:", mse)

In [ ]:
x_plot = np.linspace(X_reg.min(), X_reg.max(), 300).reshape(-1, 1)
x_plot_scaled = scaler.transform(x_plot)
y_plot = knn_reg.predict(x_plot_scaled)

plt.figure(figsize=(6, 5))
plt.scatter(X_reg[:, 0], y_reg, alpha=0.6)
plt.plot(x_plot[:, 0], y_plot, color='r', linewidth=3)
plt.xlabel("x")
plt.ylabel("y")
plt.title("KNN Regression")
plt.grid(True)
plt.show()

### Use KNN for stress-strain data. 

- Download the csv file of the stress-strain data we used in the last class. `https://raw.githubusercontent.com/huichiayu/cmse_202_802/main/data/stress_strain_T600.csv`.
- Import the data from csv file.

In [ ]:
# put your code here

In [ ]:
import pandas as pd
data_600=pd.??("stress_strain_T600.csv")

- Repeat the KNN regression method for the stress-strain data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

# =====================================================
# Prepare data
# =====================================================

X_reg = ??
y_reg = ??

# =====================================================
# Train / test split
# =====================================================
??


# =====================================================
# Scale features
# =====================================================

??

# =====================================================
# Train KNN regression
# =====================================================

knn_reg = KNeighborsRegressor(n_neighbors=5)
knn_reg.fit(X_train_scaled, y_train)

# =====================================================
# Smooth curve for plotting
# =====================================================

x_plot = np.linspace(
    X_reg.min(),
    X_reg.max(),
    300).reshape(-1, 1)

x_plot_scaled = scaler.transform(x_plot)

y_plot = knn_reg.predict(x_plot_scaled)

# =====================================================
# Plot
# =====================================================

plt.figure(figsize=(6,5))

plt.scatter(
    X_reg[:,0],
    y_reg,
    alpha=0.6,
    label="data")

plt.plot(
    x_plot[:,0],
    y_plot,
    linewidth=3, color='r',
    label="KNN regression")

plt.xlabel("Strain")
plt.ylabel("Stress")
plt.title("KNN Regression")
plt.grid(True)
plt.legend()

plt.show()

---

## 12. Please upload your completed notebook via this [Link](https://www.dropbox.com/request/jw4b8ntlojr8gkib6blh).


